# Historical experiment: inspect_model.ipynb
Archived source with outputs/execution counts cleared. Superseded by the numbered notebooks.
Original paths and experimental branches are retained for reference; this is not the supported run sequence.

In [ ]:
from collections import defaultdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_ID = "roneneldan/TinyStories-3M"

# Pin the exact model revision for reproducible experiments.
MODEL_REVISION = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"Loading {MODEL_ID}...")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

In [ ]:
model.eval()

In [ ]:
# model.parameters() counts independent Parameter objects.
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
)

In [ ]:
print("\n=== Model type ===")
print(type(model).__name__)

In [ ]:
print("\n=== Configuration ===")
print(f"Vocabulary size        : {model.config.vocab_size:,}")
print(f"Maximum sequence length: {model.config.max_position_embeddings:,}")
print(f"Transformer layers     : {model.config.num_layers}")
print(f"Hidden dimension       : {model.config.hidden_size}")
print(f"Attention heads        : {model.config.num_heads}")
print(f"Attention types        : {model.config.attention_layers}")
print(f"Activation function    : {model.config.activation_function}")

In [ ]:
print("\n=== Parameter count ===")
print(f"Total parameters       : {total_parameters:,}")
print(f"Trainable parameters   : {trainable_parameters:,}")
print(f"FP32 parameter storage : {parameter_bytes / (1024**2):.2f} MiB")
print(f"Estimated INT8 storage : {total_parameters / (1024**2):.2f} MiB")

In [ ]:
# Count parameters by major model component.
component_counts = defaultdict(int)

for name, parameter in model.named_parameters():
    parts = name.split(".")

    if len(parts) >= 3 and parts[0] == "transformer" and parts[1] == "h":
        component = f"transformer_block_{parts[2]}"
    elif name.startswith("transformer.wte"):
        component = "token_embedding"
    elif name.startswith("transformer.wpe"):
        component = "position_embedding"
    elif name.startswith("transformer.ln_f"):
        component = "final_layer_norm"
    elif name.startswith("lm_head"):
        component = "language_model_head"
    else:
        component = "other"

    component_counts[component] += parameter.numel()

In [ ]:
print("\n=== Parameters by component ===")

for component, count in component_counts.items():
    percentage = 100.0 * count / total_parameters
    print(f"{component:24s}: {count:10,d} ({percentage:6.2f}%)")

In [ ]:
# GPT-style models commonly share the token-embedding matrix with
# the output language-model head.
input_embedding = model.get_input_embeddings().weight
output_embedding = model.get_output_embeddings().weight

weights_are_tied = (
    input_embedding.data_ptr() == output_embedding.data_ptr()
)

print("\n=== Weight sharing ===")
print(f"Input/output embeddings tied: {weights_are_tied}")